# Class 3 — Rule-Based Anomaly Detector

**Deliverable:** Rule-based detector
**Date:** July 12, 2026

Goal: confirm the hour feature is genuinely useful, then combine features into one
detection rule and measure it properly.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/creditcard.csv')
print(df.shape)
print(df['Class'].value_counts())

In [ ]:
# Feature 1 - amount z-score: how unusual is this amount vs the average
mean_amount = df['Amount'].mean()
std_amount  = df['Amount'].std()
df['amount_zscore'] = (df['Amount'] - mean_amount) / std_amount

# Feature 2 - hour of day: Time is seconds since first transaction, not clock time
df['hour'] = (df['Time'] / 3600) % 24

# Feature 3 - is_night: midnight to 6am
df['is_night'] = ((df['hour'] >= 0) & (df['hour'] <= 6)).astype(int)

print(df[['Amount', 'amount_zscore', 'hour', 'is_night']].head())

In [ ]:
def evaluate(flag_series, label='detector'):
    """Precision/recall for a boolean flag column against the true Class label."""
    caught  = df[(flag_series) & (df['Class'] == 1)].shape[0]
    flagged = df[flag_series].shape[0]
    total   = df[df['Class'] == 1].shape[0]
    recall    = caught / total * 100 if total else 0
    precision = caught / flagged * 100 if flagged else 0
    print(f"{label}")
    print(f"  fraud caught : {caught} / {total}")
    print(f"  total flagged: {flagged}")
    print(f"  recall       : {recall:.2f}%")
    print(f"  precision    : {precision:.2f}%")
    return recall, precision

## Step 1 — Is night time actually a real signal?

In [ ]:
night_fraud_rate = df[df['is_night'] == 1]['Class'].mean() * 100
day_fraud_rate   = df[df['is_night'] == 0]['Class'].mean() * 100

print("Fraud rate at night:", night_fraud_rate)
print("Fraud rate during day:", day_fraud_rate)

A transaction at night is roughly **4x more likely** to be fraud than one during the day.
This confirms the hour feature from Class 2 was a real signal, not just a nice-looking chart.

## Step 2 — The combined rule

In [ ]:
# Flag if the amount is very unusual OR it happened at night,
# but only if the amount is above the median (kills a huge number of tiny false positives)
df['flagged'] = (
    ((df['amount_zscore'] > 3) | (df['is_night'] == 1))
    & (df['Amount'] > df['Amount'].median())
)

## Step 3 — Measuring it

In [ ]:
evaluate(df['flagged'], 'Class 3 combined rule')

Recorded result: **49 fraud caught out of 492**, **13,503 flagged**.

- Recall **9.96%** — up from 2.2% in Class 2, almost 5x
- Precision **~0.36%** — still terrible

## Where it falls short

Flagging 13,503 transactions to catch 49 real frauds means ~275 false alarms per real
catch. No business would use this.

The deeper problem: an OR rule cannot express how features *interact*. A \$150
transaction at 2am might be normal for one customer and highly suspicious for another.
A rule treats every condition independently. That limitation is exactly what the
Isolation Forest in Class 4 is meant to fix.